# 4. Ejecutar Pipeline Silver

Propósito: Ejecutar el pipeline silver completo (quality + transform + parquet).

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6
JAVA_HOME: C:\Program Files\Java\jdk-21


In [2]:
from app.utils.spark import SparkClient
spark_client = SparkClient()
spark = spark_client.get_session()

In [3]:
# Metodo C (= "Opcion C" de INSTRUCCIONES_EJECUCION.txt): municipios="drop".
# Lima=C / resto=G, una fila por nombre de municipio (la del menor SEC_EJEC).
# Equivale a:  python main.py silver --drop --municipios drop
MUNICIPIOS = "drop"

from app.silver import quality
stage = quality.fix_all(spark, municipios=MUNICIPIOS)
print(f"Quality fixes completados (municipios={MUNICIPIOS})")

2026-06-18 21:41:19,836 - INFO - Iniciando correcciones de calidad para todos los datasets
2026-06-18 21:41:19,836 - INFO - Procesando dataset SIAF - Ingreso
2026-06-18 21:42:01,836 - INFO - ingreso_unified: 3104689 filas después de correcciones
2026-06-18 21:42:01,838 - INFO - Procesando dataset SISMEPRE
2026-06-18 21:42:03,101 - INFO - rentas_preguntas: 836 filas después de correcciones
2026-06-18 21:42:04,164 - INFO - rentas_formulario: 98 filas después de correcciones
2026-06-18 21:42:07,914 - INFO - rentas_esat_estadistica_atm: 134144 filas después de correcciones
2026-06-18 21:42:10,480 - INFO - rentas_respuestas: 250179 filas después de correcciones
2026-06-18 21:42:11,132 - INFO - rentas_ano_aplicacion: 26 filas después de correcciones
2026-06-18 21:42:54,012 - INFO - Guardado: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6\data\silver\stage\ingreso_unified.parquet
2026-06-18 21:42:54,012 - INFO - Etiquetando categorías municipales (modo=drop)
2026-06-18 21:42:59,616 - INF

Quality fixes completados (municipios=drop)


In [4]:
from app.silver import transforms
dims, facts = transforms.build_all(spark, stage, municipios=MUNICIPIOS)
print(f"Dimensiones: {len(dims)}, Hechos: {len(facts)}")

2026-06-18 21:44:38,961 - INFO - Construyendo dims + hechos para silver parquet (municipios=drop)
2026-06-18 21:44:38,967 - INFO - Construyendo dimensiones del modelo estrella (municipios=drop)
2026-06-18 21:45:47,981 - INFO - Dimensiones construidas: 14 dimensiones
2026-06-18 21:45:48,247 - INFO - Construyendo DIM_PREGUNTA_SISMEPRE
2026-06-18 21:45:48,501 - INFO - Construyendo tablas de hechos
2026-06-18 21:46:52,861 - INFO - Tablas de hechos construidas: 3 fact tables


Dimensiones: 15, Hechos: 3


In [5]:
from app.silver.parquet_loader import save_all_to_parquet
from pathlib import Path
save_all_to_parquet(dims, facts, Path("data/silver"))
print("Silver parquet guardados en data/silver/")

2026-06-18 21:48:26,471 - INFO - Saved DIM_TIEMPO to data\silver\DIM_TIEMPO.parquet
2026-06-18 21:49:05,737 - INFO - Saved DIM_EJECUTORA to data\silver\DIM_EJECUTORA.parquet
2026-06-18 21:49:55,646 - INFO - Saved DIM_UBIGEO to data\silver\DIM_UBIGEO.parquet
2026-06-18 21:49:58,580 - INFO - Saved DIM_NIVEL_GOBIERNO to data\silver\DIM_NIVEL_GOBIERNO.parquet
2026-06-18 21:50:00,158 - INFO - Saved DIM_SECTOR to data\silver\DIM_SECTOR.parquet
2026-06-18 21:50:01,669 - INFO - Saved DIM_PLIEGO to data\silver\DIM_PLIEGO.parquet
2026-06-18 21:50:05,260 - INFO - Saved DIM_RUBRO to data\silver\DIM_RUBRO.parquet
2026-06-18 21:50:08,779 - INFO - Saved DIM_TIPO_RECURSO to data\silver\DIM_TIPO_RECURSO.parquet
2026-06-18 21:50:11,607 - INFO - Saved DIM_FUENTE_FINANCIAMIENTO to data\silver\DIM_FUENTE_FINANCIAMIENTO.parquet
2026-06-18 21:50:17,907 - INFO - Saved DIM_GENERICA to data\silver\DIM_GENERICA.parquet
2026-06-18 21:50:23,523 - INFO - Saved DIM_ESPECIFICA to data\silver\DIM_ESPECIFICA.parquet
20

Silver parquet guardados en data/silver/


In [6]:
import os
for f in sorted(os.listdir("data/silver")):
    if f.endswith(".parquet"):
        df = spark.read.parquet(f"data/silver/{f}")
        print(f"{f}: {df.count():,} filas")

DIM_ANIO_APLICACION.parquet: 14 filas
DIM_EJECUTORA.parquet: 1,769 filas
DIM_ESPECIFICA.parquet: 74 filas
DIM_FORMULARIO_SISMEPRE.parquet: 16 filas
DIM_FUENTE_FINANCIAMIENTO.parquet: 4 filas
DIM_GENERICA.parquet: 59 filas
DIM_NIVEL_GOBIERNO.parquet: 1 filas
DIM_PLIEGO.parquet: 1 filas
DIM_PREGUNTA_RENAMU.parquet: 532,538 filas
DIM_PREGUNTA_SISMEPRE.parquet: 236 filas
DIM_RUBRO.parquet: 6 filas
DIM_SECTOR.parquet: 1 filas
DIM_TIEMPO.parquet: 67 filas
DIM_TIPO_RECURSO.parquet: 55 filas
DIM_UBIGEO.parquet: 1,891 filas
FACT_FORMULARIO_SISMEPRE.parquet: 230,942 filas
FACT_INGRESO.parquet: 2,431,820 filas
FACT_RENAMU.parquet: 12,770,291 filas
